# Домашнее задание № 2. Мешок слов

## Задание 1 (3 балла)

У векторайзеров в sklearn есть встроенная токенизация на регулярных выражениях. Найдите способо заменить её на кастомную токенизацию

Обучите векторайзер с дефолтной токенизацией и с токенизацией razdel.tokenize. Обучите классификатор (любой) с каждым из векторизаторов. Сравните метрики и выберете победителя. 

(в вашей тетрадке должен быть код обучения и все метрики; если вы сдаете в .py файлах то сохраните полученные метрики в отдельном файле или в комментариях)

In [190]:
import pandas as pd

In [191]:
data = pd.read_csv('labeled.csv')

In [192]:
data

,comment,toxic
0,"Верблюдов-то за что? Дебилы, бл...\n",1.0
1,"Хохлы, это отдушина затюканого россиянина, мол...",1.0
2,Собаке - собачья смерть\n,1.0
3,"Страницу обнови, дебил. Это тоже не оскорблени...",1.0
4,"тебя не убедил 6-страничный пдф в том, что Скр...",1.0
...,...,...
14407,Вонючий совковый скот прибежал и ноет. А вот и...,1.0
14408,А кого любить? Гоблина тупорылого что-ли? Или ...,1.0
14409,"Посмотрел Утомленных солнцем 2. И оказалось, ч...",0.0
14410,КРЫМОТРЕД НАРУШАЕТ ПРАВИЛА РАЗДЕЛА Т.К В НЕМ Н...,1.0


In [193]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14412 entries, 0 to 14411
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   comment  14412 non-null  object 
 1   toxic    14412 non-null  float64
dtypes: float64(1), object(1)
memory usage: 225.3+ KB


In [194]:
from sklearn.model_selection import train_test_split
X = data['comment']
y = data['toxic']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [195]:
from sklearn.feature_extraction.text import CountVectorizer
vec = CountVectorizer()
print(vec.token_pattern)


(?u)\b\w\w+\b


In [196]:
help(CountVectorizer)

Help on class CountVectorizer in module sklearn.feature_extraction.text:

class CountVectorizer(_VectorizerMixin, sklearn.base.BaseEstimator)
 |  CountVectorizer(*, input='content', encoding='utf-8', decode_error='strict', strip_accents=None, lowercase=True, preprocessor=None, tokenizer=None, stop_words=None, token_pattern='(?u)\\b\\w\\w+\\b', ngram_range=(1, 1), analyzer='word', max_df=1.0, min_df=1, max_features=None, vocabulary=None, binary=False, dtype=<class 'numpy.int64'>)
 |  
 |  Convert a collection of text documents to a matrix of token counts.
 |  
 |  This implementation produces a sparse representation of the counts using
 |  scipy.sparse.csr_matrix.
 |  
 |  If you do not provide an a-priori dictionary and you do not use an analyzer
 |  that does some kind of feature selection then the number of features will
 |  be equal to the vocabulary size found by analyzing the data.
 |  
 |  For an efficiency comparison of the different feature extractors, see
 |  :ref:`sphx_glr_au

In [197]:
from razdel import tokenize

def razdel_tokenizer(text):
    return [t.text for t in tokenize(text)]
vectorizer_razdel = CountVectorizer(
    tokenizer=razdel_tokenizer,
    token_pattern=None 
)



In [198]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

def evaluate_vectorizer(vectorizer, name):
    print(f"\n=== {name} ===")

    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    model = LogisticRegression(max_iter=2000, C=2.0, class_weight='balanced')
    model.fit(X_train_vec, y_train)

    y_pred = model.predict(X_test_vec)

    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)

    print(f"Accuracy: {acc:.4f}, F1: {f1:.4f}")
    return {"name": name, "accuracy": acc, "f1": f1}

results = [
    evaluate_vectorizer(CountVectorizer(), "Default tokenizer"),
    evaluate_vectorizer(vectorizer_razdel, "Razdel tokenizer"),
]



=== Default tokenizer ===
Accuracy: 0.8651, F1: 0.7960

=== Razdel tokenizer ===
Accuracy: 0.8647, F1: 0.7967


## Задание 2 (3 балла)

Обучите 2 любых разных классификатора из семинара. Предскажите токсичность для текстов из тестовой выборки (используйте одну и ту же выборку для обоих классификаторов) и найдите 10 самых токсичных для каждого из классификаторов. Сравните получаемые тексты - какие тексты совпадают, какие отличаются, правда ли тексты токсичные?

Требования к моделям:   
а) один классификатор должен использовать CountVectorizer, другой TfidfVectorizer  
б) у векторазера должны быть вручную заданы как минимум 5 параметров (можно ставить разные параметры tfidfvectorizer и countvectorizer)  
в) у классификатора должно быть задано вручную как минимум 2 параметра (по возможности)  
г)  f1 мера каждого из классификаторов должна быть минимум 0.75  

*random_seed не считается за параметр

In [199]:
count_vect = CountVectorizer(
    lowercase=True,        
    analyzer='word',       
    ngram_range=(1,2),     
    max_df=0.95,           
    min_df=2,              
    max_features=25000    
)

In [200]:
from sklearn.feature_extraction.text import  TfidfVectorizer


tfidf_vect = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1,3),
    min_df=3,
    max_df=0.9,
    sublinear_tf=True,
    norm='l2'
)

In [201]:
X_train_count = count_vect.fit_transform(X_train)
X_test_count  = count_vect.transform(X_test)



In [202]:
X_train_tfidf = tfidf_vect.fit_transform(X_train)
X_test_tfidf  = tfidf_vect.transform(X_test)

In [203]:
print("Shapes:")
print(" Count X_train:", X_train_count.shape, "X_test:", X_test_count.shape)



Shapes:
 Count X_train: (11529, 25000) X_test: (2883, 25000)


In [204]:
print(" TF-IDF X_train:", X_train_tfidf.shape, "X_test:", X_test_tfidf.shape)

 TF-IDF X_train: (11529, 21126) X_test: (2883, 21126)


In [205]:
from sklearn.naive_bayes import MultinomialNB
clf_nb = MultinomialNB(alpha=0.5, fit_prior=True) 

In [206]:
from sklearn.linear_model import LogisticRegression
clf_lr = LogisticRegression(
    C=2.0,
    solver='liblinear',
    class_weight='balanced',
    max_iter=2000
)


In [207]:
clf_nb.fit(X_train_count, y_train)


,alpha,0.5
,force_alpha,True
,fit_prior,True
,class_prior,None


In [208]:
clf_lr.fit(X_train_tfidf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,2.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'liblinear'
,max_iter,2000
,multi_class,'deprecated'


In [209]:
from sklearn.metrics import classification_report

pred_nb = clf_nb.predict(X_test_count)

report_nb = classification_report(y_test, pred_nb)



In [210]:
pred_lr = clf_lr.predict(X_test_tfidf)

report_lr = classification_report(y_test, pred_lr)

In [211]:
print(report_nb)

              precision    recall  f1-score   support

         0.0       0.88      0.93      0.91      1918
         1.0       0.85      0.75      0.79       965

    accuracy                           0.87      2883
   macro avg       0.86      0.84      0.85      2883
weighted avg       0.87      0.87      0.87      2883



In [212]:
print(report_lr)

              precision    recall  f1-score   support

         0.0       0.90      0.90      0.90      1918
         1.0       0.80      0.80      0.80       965

    accuracy                           0.86      2883
   macro avg       0.85      0.85      0.85      2883
weighted avg       0.86      0.86      0.86      2883



In [213]:
import numpy as np


topk = 10

nb_scores = clf_nb.predict_proba(X_test_count)[:, 1]
nb_idx = np.argsort(nb_scores)[-topk:][::-1]
top10_nb_texts = [(i, X_test.iloc[i], nb_scores[i], y_test.iloc[i]) for i in nb_idx]


lr_scores = clf_lr.predict_proba(X_test_tfidf)[:, 1] 
lr_idx = np.argsort(lr_scores)[-topk:][::-1]
top10_lr_texts = [(i, X_test.iloc[i], lr_scores[i], y_test.iloc[i]) for i in lr_idx]


In [214]:
print("\n=== Top-10 most toxic by NaiveBayes ===")
for rank, (i, txt, score, true_label) in enumerate(top10_nb_texts, 1):
    print(f"{rank:2d}. idx={i} score={score:.4f} true={true_label}\n   {txt[:300].replace(chr(10),' ')}\n")



=== Top-10 most toxic by NaiveBayes ===
 1. idx=1957 score=1.0000 true=1.0
   БЕЛАРУСЬ, БЛЯТЬ, БЕЛАРУСЬ. СПИДОРСВИН, БЛЯТЬ. НЕПРОБИВАЕМАЯ ХОХЛИНА, СУКА. Какие-же хохлы дененераты, пиздец просто. 

 2. idx=2293 score=1.0000 true=1.0
   черт опущенный Гомикадзе Би опущенный гей на ебало ссали хуй сосать черт и пиздабол чисто мразь в мазке. Одна ложь и грязь. пиздаболище ПиськаМазе х15р 150р Угомонись, разоришь пригожина 

 3. idx=913 score=1.0000 true=1.0
   Что? Какой денежный петух? Нет у хохлов своих сказок, коммуняки потом навыдумали хохлам культурку. А по факту то нация свинопасов и быдла, не городская. В том же Львове жили в основном поляки и евреи - городское население, а хохлы так. Быдло и селюки. 

 4. idx=1538 score=1.0000 true=0.0
   Возьмём как пример Россию, западноевропейские страны и США. Идёт метисация, сознательная политика замещения белого населения на пришлое черно-коричневое. Идёт создание новой расы метисов, исламизация и почернение. В крупных городах половина насел

In [215]:
print("\n=== Top-10 most toxic by LogisticRegression ===")
for rank, (i, txt, score, true_label) in enumerate(top10_lr_texts, 1):
    print(f"{rank:2d}. idx={i} score={score:.4f} true={true_label}\n   {txt[:300].replace(chr(10),' ')}\n")


=== Top-10 most toxic by LogisticRegression ===
 1. idx=590 score=0.9997 true=1.0
   Пидр ты, эксклюзивный. 

 2. idx=344 score=0.9979 true=1.0
   По мексикански Флаг: Ублюдок, мать твою, а ну иди сюда говно собачье, решил меня поднять? Ты, засранец вонючий, мать твою, а? Ну иди сюда, попробуй меня поднять, я тебя сам подниму ублюдок, онанист чертов, будь ты проклят, иди идиот, трахать тебя и всю семью, говно собачье, жлоб вонючий, дерьмо, сук

 3. idx=2666 score=0.9976 true=1.0
   Какие блять передергивания? Ты дебил блять зашел на шок-доску и удивляешься что над тобой издеваются. Тут нет твоих друзей, рачье тупорылое, тут тебя все ненавидят. Как же печет от таких необучаемых ебланов. Ты ковбой, твою жену ебут где-то нахуй, а дети гибнут на Украине. Понял, быдло ты ебаное? 

 4. idx=208 score=0.9972 true=1.0
   Блядь абу нахуй ссылай этих дегенератов в фаг, всем похуй на их шлюх 

 5. idx=363 score=0.9966 true=1.0
   Да ты пидор какляцкий. В жопу тебя ебут. 

 6. idx=13 score=0.9962 

In [216]:
set_nb = set(nb_idx)
set_lr = set(lr_idx)

print("Пересечение топ-10 индексов:", sorted(set_nb & set_lr))
print("Только NaiveBayes:", sorted(set_nb - set_lr))
print("Только LogisticRegression:", sorted(set_lr - set_nb))

Пересечение топ-10 индексов: [344, 1957]
Только NaiveBayes: [279, 913, 1051, 1538, 2293, 2322, 2541, 2698]
Только LogisticRegression: [13, 208, 363, 590, 1328, 1537, 2057, 2666]


In [217]:
true_in_nb_top10 = sum(1 for (_, _, _, true) in top10_nb_texts if true==1)
true_in_lr_top10 = sum(1 for (_, _, _, true) in top10_lr_texts if true==1)

print(f"\nВ топ-10 NaiveBayes реально токсичных (на основе лэйбла) = {true_in_nb_top10}/10")
print(f"В топ-10 LogisticRegression реально токсичных (на основе лэйбла) = {true_in_lr_top10}/10")


В топ-10 NaiveBayes реально токсичных (на основе лэйбла) = 8/10
В топ-10 LogisticRegression реально токсичных (на основе лэйбла) = 10/10


### Тексты, которые NaiveBayes пометил как токсичные, хотя в датасете они не размечены таковыми, содержат много «токсичных» слов и ругательств, характерных для токсичных комментариев. И оба этих текста, на мой взгляд, можно считать таксичными.

## Задание 3 (4 балла - 1 балл за каждый классификатор)

Для классификаторов Logistic Regression, Decision Trees, Naive Bayes, RandomForest найдите способ извлечь важность признаков для предсказания токсичного класса. Сопоставьте полученные числа со словами (или нграммами) в словаре и найдите топ - 5 "токсичных" слов для каждого из классификаторов. 

Важное требование: в топе не должно быть стоп-слов. Для этого вам нужно будет правильным образом настроить векторизацию. 
Также как и в предыдущем задании у классификаторов должно быть задано вручную как минимум 2 параметра (по возможности, f1 мера каждого из классификаторов должна быть минимум 0.75

In [218]:
data = pd.read_csv('labeled.csv')
X = data['comment']
y = data['toxic']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [224]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
russian_stopwords = set(stopwords.words('russian'))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/kseniazavyalova/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [230]:
russian_stopwords.add("тебе")

In [231]:
vectorizer = TfidfVectorizer(
    stop_words= list(russian_stopwords),
    lowercase=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
feature_names = vectorizer.get_feature_names_out()


In [ ]:
feature_names

array(['a', 'adobe', 'age', ..., 'ёж', 'ёлка', 'ёмкость'], dtype=object)

In [96]:
def top_k_pairs(names, scores, k=5):
    order = np.argsort(scores)[::-1]
    return [(names[i], float(scores[i])) for i in order[:k]]

In [98]:
def print_top(title, pairs):
    print(f"\nТоп «токсичных» признаков — {title}:")
    for i, (feat, val) in enumerate(pairs, 1):
        print(f"{i:>2}. {feat:30s}  {val: .4f}")

In [180]:
def get_top_toxic_words(model, feature_names, model_name, n_top=5):
    
    if model_name == 'Logistic Regression':
        importances = model.coef_[0]
        
    elif model_name == 'Decision Tree' or model_name == 'Random Forest':
        importances = model.feature_importances_
        
    elif model_name == 'Naive Bayes':
        log_prob_toxic = model.feature_log_prob_[1]
        log_prob_non_toxic = model.feature_log_prob_[0] 
        importances = log_prob_toxic - log_prob_non_toxic
    
    top_indices = np.argsort(importances)[-n_top:][::-1]
    
    top_words = [(feature_names[i], importances[i]) for i in top_indices]
    
    return top_words

In [233]:
models = {}
results = {}
top_words_dict = {}

In [234]:

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score

print("=" * 50)
print("1. Logistic Regression with GridSearchCV")
print("-" * 50)

lr_base = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42, solver='liblinear')

param_grid = {
    'C': [0.5, 1.0, 2.0, 5.0],          
    'penalty': ['l2'],            
    'solver': ['liblinear']             
}

f1_scorer = make_scorer(f1_score)


grid_lr = GridSearchCV(
    lr_base,
    param_grid,
    scoring=f1_scorer,
    cv=5,
    n_jobs=-1
)


grid_lr.fit(X_train_tfidf, y_train)

best_lr = grid_lr.best_estimator_
print("Best parameters:", grid_lr.best_params_)

y_pred_lr = best_lr.predict(X_test_tfidf)
f1_lr = f1_score(y_test, y_pred_lr)
print(f"\nF1-score on test set: {f1_lr:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Non-toxic', 'Toxic']))

models['Logistic Regression'] = best_lr
results['Logistic Regression'] = f1_lr

top_words = get_top_toxic_words(best_lr, feature_names, 'Logistic Regression')
top_words_dict['Logistic Regression'] = top_words

print("\nТоп-5 токсичных слов:")
for word, importance in top_words:
    print(f"  {word}: {importance:.4f}")


1. Logistic Regression with GridSearchCV
--------------------------------------------------
Best parameters: {'C': 2.0, 'penalty': 'l2', 'solver': 'liblinear'}

F1-score on test set: 0.7835

Classification Report:
              precision    recall  f1-score   support

   Non-toxic       0.88      0.91      0.90      1918
       Toxic       0.81      0.76      0.78       965

    accuracy                           0.86      2883
   macro avg       0.85      0.83      0.84      2883
weighted avg       0.86      0.86      0.86      2883


Топ-5 токсичных слов:
  хохлы: 5.9427
  хохлов: 5.7150
  нахуй: 4.5177
  пиздец: 4.2656
  блядь: 4.2236


In [ ]:
print("=" * 50)
print("2. Naive Bayes with GridSearchCV")
print("-" * 50)


nb_base = MultinomialNB()
param_grid_nb = {
    'alpha': [1.0, 0.5, 0.1, 0.05, 0.01], 
    'fit_prior': [True, False] 
}
grid_nb = GridSearchCV(
    nb_base,
    param_grid_nb,
    scoring=f1_scorer,
    cv=5,
    n_jobs=-1
)

grid_nb.fit(X_train_tfidf, y_train)

best_nb = grid_nb.best_estimator_
print("Best parameters:", grid_nb.best_params_)

y_pred_nb = best_nb.predict(X_test_tfidf)
f1_nb = f1_score(y_test, y_pred_nb)

print(f"\nF1-score on test set: {f1_nb:.4f}")
if f1_nb < 0.75:
    print("!!! ВНИМАНИЕ: F1-score ниже требуемого порога 0.75 !!!")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb, target_names=['Non-toxic', 'Toxic']))

models['Naive Bayes'] = best_nb
results['Naive Bayes'] = f1_nb

top_words_nb = get_top_toxic_words(best_nb, feature_names, 'Naive Bayes')
top_words_dict['Naive Bayes'] = top_words_nb

print("\nТоп-5 токсичных слов:")
for word, importance in top_words_nb:
    print(f"  {word}: {importance:.4f}")




2. Naive Bayes with GridSearchCV
--------------------------------------------------
Best parameters: {'alpha': 0.1, 'fit_prior': False}

F1-score on test set: 0.7853

Classification Report:
              precision    recall  f1-score   support

   Non-toxic       0.88      0.92      0.90      1918
       Toxic       0.82      0.75      0.79       965

    accuracy                           0.86      2883
   macro avg       0.85      0.84      0.84      2883
weighted avg       0.86      0.86      0.86      2883


Топ-5 токсичных слов:
  хохлов: 6.1355
  сука: 5.4307
  дебил: 5.2668
  хохол: 5.1491
  русня: 4.8649


In [ ]:
print("=" * 50)
print("3. Random Forest with GridSearchCV")
print("-" * 50)

rf_base = RandomForestClassifier(
    class_weight='balanced',
    random_state=42
)

param_grid_rf = {
    'n_estimators': [150, 250],           
    'max_depth': [30, 50, None],           
    'min_samples_leaf': [3, 5]             
}

grid_rf = GridSearchCV(
    rf_base,
    param_grid_rf,
    scoring=f1_scorer,
    cv=3,
    verbose=2 
)

grid_rf.fit(X_train_tfidf, y_train)

best_rf = grid_rf.best_estimator_
print("\nBest parameters:", grid_rf.best_params_)

y_pred_rf = best_rf.predict(X_test_tfidf)
f1_rf = f1_score(y_test, y_pred_rf)

print(f"\nF1-score on test set: {f1_rf:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Non-toxic', 'Toxic']))

models['Random Forest'] = best_rf
results['Random Forest'] = f1_rf

top_words_rf = get_top_toxic_words(best_rf, feature_names, 'Random Forest')
top_words_dict['Random Forest'] = top_words_rf

print("\nТоп-5 токсичных слов:")
for word, importance in top_words_rf:
    print(f"  {word}: {importance:.4f}")

3. Random Forest with GridSearchCV
--------------------------------------------------
Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END .max_depth=30, min_samples_leaf=3, n_estimators=150; total time=   5.9s
[CV] END .max_depth=30, min_samples_leaf=3, n_estimators=150; total time=   5.8s
[CV] END .max_depth=30, min_samples_leaf=3, n_estimators=150; total time=   5.7s
[CV] END .max_depth=30, min_samples_leaf=3, n_estimators=250; total time=   9.4s
[CV] END .max_depth=30, min_samples_leaf=3, n_estimators=250; total time=   9.6s
[CV] END .max_depth=30, min_samples_leaf=3, n_estimators=250; total time=   9.4s
[CV] END .max_depth=30, min_samples_leaf=5, n_estimators=150; total time=   5.5s
[CV] END .max_depth=30, min_samples_leaf=5, n_estimators=150; total time=   5.7s
[CV] END .max_depth=30, min_samples_leaf=5, n_estimators=150; total time=   5.6s
[CV] END .max_depth=30, min_samples_leaf=5, n_estimators=250; total time=   9.3s
[CV] END .max_depth=30, min_samples_leaf=5,

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer(ngram_range=(1, 2), max_df=0.8, min_df=5)),
    ('selector', SelectKBest(chi2, k=10000)),
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])

pipeline.fit(X_train, y_train)

param_grid_pipeline = {
    'selector__k': [7000, 10000, 15000],

    'classifier__n_estimators': [250, 400],
    'classifier__max_depth': [80, None],
    'classifier__min_samples_leaf': [1, 2]
}
grid_pipeline = GridSearchCV(pipeline, param_grid_pipeline, scoring=f1_scorer, cv=3, verbose=2, n_jobs=-1)
grid_pipeline.fit(X_train, y_train)


In [ ]:
import re
import pymorphy3
from nltk.corpus import stopwords
import pandas as pd 
russian_stopwords = stopwords.words("russian")
custom_stopwords = [
    'это', 'как', 'так', 'и', 'в', 'над', 'к', 'до', 'не', 'на', 'но', 'за', 'то', 'с', 'ли',
    'а', 'во', 'от', 'со', 'для', 'о', 'же', 'ну', 'вы', 'бы', 'что', 'кто', 'он', 'она',
    'очень', 'спасибо', 'знаю', 'тебе', 'например', 'вообще', 'просто', 'можно', 'ты', 'тебя'
]
russian_stopwords.extend(custom_stopwords)
morph = pymorphy3.MorphAnalyzer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^а-яa-z\s]', ' ', text)
    tokens = text.split()
    lemmatized_tokens = [morph.parse(token)[0].normal_form for token in tokens if token not in russian_stopwords]
    return " ".join(lemmatized_tokens)

In [264]:
data['processed_comment'] = data['comment'].apply(preprocess_text)

print(data[['comment', 'processed_comment']].head())

                                             comment  \
0               Верблюдов-то за что? Дебилы, бл...\n   
1  Хохлы, это отдушина затюканого россиянина, мол...   
2                          Собаке - собачья смерть\n   
3  Страницу обнови, дебил. Это тоже не оскорблени...   
4  тебя не убедил 6-страничный пдф в том, что Скр...   

                                   processed_comment  
0                                   верблюд дебил бл  
1  хохол отдушина затюканый россиянин мол вон хох...  
2                              собака собачий смерть  
3  страница обновить дебил оскорбление доказать ф...  
4  убедить страничный пдф скрипаль отравить росси...  


In [ ]:
from sklearn.model_selection import train_test_split
X = data['processed_comment'] 
y = data['toxic'] 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [262]:
grid_pipeline.fit(X_train, y_train)

Fitting 3 folds for each of 24 candidates, totalling 72 fits
[CV] END classifier__max_depth=80, classifier__min_samples_leaf=1, classifier__n_estimators=250, selector__k=7000; total time=   9.5s
[CV] END classifier__max_depth=80, classifier__min_samples_leaf=1, classifier__n_estimators=250, selector__k=7000; total time=  10.3s
[CV] END classifier__max_depth=80, classifier__min_samples_leaf=1, classifier__n_estimators=250, selector__k=15000; total time=  10.1s
[CV] END classifier__max_depth=80, classifier__min_samples_leaf=1, classifier__n_estimators=250, selector__k=7000; total time=  10.5s
[CV] END classifier__max_depth=80, classifier__min_samples_leaf=1, classifier__n_estimators=250, selector__k=10000; total time=  10.4s
[CV] END classifier__max_depth=80, classifier__min_samples_leaf=1, classifier__n_estimators=250, selector__k=10000; total time=  10.3s
[CV] END classifier__max_depth=80, classifier__min_samples_leaf=1, classifier__n_estimators=250, selector__k=10000; total time=  10.

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'classifier__max_depth': [80, None], 'classifier__min_samples_leaf': [1, 2], 'classifier__n_estimators': [250, 400], 'selector__k': [7000, 10000, ...]}"
,scoring,make_scorer(f...hod='predict')
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


In [ ]:

print("\n" + "="*50)
print("Оценка лучшей модели на тестовых данных")
print("-" * 50)

best_pipeline = grid_pipeline.best_estimator_

print("Лучшие найденные параметры:", grid_pipeline.best_params_)
print("Лучший F1-score на кросс-валидации:", grid_pipeline.best_score_)

y_pred = best_pipeline.predict(X_test)

final_f1_score = f1_score(y_test, y_pred)
print(f"\nИтоговый F1-score на тестовом наборе: {final_f1_score:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Non-toxic', 'Toxic']))


print("\n" + "="*50)
print("Анализ признаков лучшей модели")
print("-" * 50)

vectorizer = best_pipeline.named_steps['vectorizer']
selector = best_pipeline.named_steps['selector']
classifier = best_pipeline.named_steps['classifier']

all_feature_names = np.array(vectorizer.get_feature_names_out())

selected_features_mask = selector.get_support()

selected_feature_names = all_feature_names[selected_features_mask]

importances = classifier.feature_importances_

feature_importances_dict = dict(zip(selected_feature_names, importances))
sorted_toxic_words = sorted(feature_importances_dict.items(), key=lambda item: item[1], reverse=True)

print("Топ-5 токсичных слов (по версии лучшей модели из Pipeline):")
for word, importance in sorted_toxic_words[:5]:
    print(f"  {word}: {importance:.4f}")


Оценка лучшей модели на тестовых данных
--------------------------------------------------
Лучшие найденные параметры: {'classifier__max_depth': None, 'classifier__min_samples_leaf': 2, 'classifier__n_estimators': 400, 'selector__k': 7000}
Лучший F1-score на кросс-валидации: 0.7161993010076367

Итоговый F1-score на тестовом наборе: 0.7274

Classification Report:
              precision    recall  f1-score   support

   Non-toxic       0.87      0.85      0.86      1918
       Toxic       0.72      0.74      0.73       965

    accuracy                           0.82      2883
   macro avg       0.79      0.80      0.79      2883
weighted avg       0.82      0.82      0.82      2883


Анализ признаков лучшей модели
--------------------------------------------------
Топ-5 токсичных слов (по версии лучшей модели из Pipeline):
  хохол: 0.0165
  год: 0.0154
  русский: 0.0120
  блядь: 0.0111
  тупой: 0.0102


In [266]:
top_words_rf_1 = sorted_toxic_words[:5]

In [270]:
top_words_dict['Random Forest'] = top_words_rf_1


In [ ]:

from sklearn.tree import DecisionTreeClassifier

print("=" * 50)
print("4. Decision Tree with GridSearchCV")
print("-" * 50)

pipeline_dt = Pipeline([
    ('vectorizer', TfidfVectorizer(ngram_range=(1, 2), max_df=0.8, min_df=5)),
    ('selector', SelectKBest(chi2)),

    ('classifier', DecisionTreeClassifier(class_weight='balanced', random_state=42))
])

param_grid_dt = {
    'selector__k': [5000, 7000, 10000],
    'classifier__criterion': ['gini', 'entropy'],
    'classifier__max_depth': [30, 50, 80, None],
    'classifier__min_samples_leaf': [2, 5, 10],
    'classifier__min_samples_split': [2, 10, 20]
}

grid_dt = GridSearchCV(
    pipeline_dt, 
    param_grid_dt, 
    scoring=f1_scorer, 
    cv=3, 
    verbose=2, 
    n_jobs=-1
)

grid_dt.fit(X_train, y_train)

print("\n" + "="*50)
print("Оценка лучшей модели Decision Tree на тестовых данных")
print("-" * 50)

best_pipeline_dt = grid_dt.best_estimator_

print("Лучшие найденные параметры:", grid_dt.best_params_)
print("Лучший F1-score на кросс-валидации:", grid_dt.best_score_)

y_pred_dt = best_pipeline_dt.predict(X_test)
final_f1_score_dt = f1_score(y_test, y_pred_dt)
print(f"\nИтоговый F1-score на тестовом наборе: {final_f1_score_dt:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt, target_names=['Non-toxic', 'Toxic']))

print("\n" + "="*50)
print("Анализ признаков лучшей модели Decision Tree")
print("-" * 50)

vectorizer_dt = best_pipeline_dt.named_steps['vectorizer']
selector_dt = best_pipeline_dt.named_steps['selector']
classifier_dt = best_pipeline_dt.named_steps['classifier']

all_feature_names_dt = np.array(vectorizer_dt.get_feature_names_out())
selected_features_mask_dt = selector_dt.get_support()
selected_feature_names_dt = all_feature_names_dt[selected_features_mask_dt]

importances_dt = classifier_dt.feature_importances_

feature_importances_dict_dt = dict(zip(selected_feature_names_dt, importances_dt))
sorted_toxic_words_dt = sorted(feature_importances_dict_dt.items(), key=lambda item: item[1], reverse=True)

print("Топ-5 токсичных слов (по версии Decision Tree):")
for word, importance in sorted_toxic_words_dt[:5]:
    print(f"  {word}: {importance:.4f}")

4. Decision Tree with GridSearchCV
--------------------------------------------------
Fitting 3 folds for each of 216 candidates, totalling 648 fits
[CV] END classifier__criterion=gini, classifier__max_depth=30, classifier__min_samples_leaf=2, classifier__min_samples_split=2, selector__k=7000; total time=   1.3s
[CV] END classifier__criterion=gini, classifier__max_depth=30, classifier__min_samples_leaf=2, classifier__min_samples_split=2, selector__k=5000; total time=   1.3s
[CV] END classifier__criterion=gini, classifier__max_depth=30, classifier__min_samples_leaf=2, classifier__min_samples_split=2, selector__k=10000; total time=   1.3s
[CV] END classifier__criterion=gini, classifier__max_depth=30, classifier__min_samples_leaf=2, classifier__min_samples_split=2, selector__k=5000; total time=   1.3s
[CV] END classifier__criterion=gini, classifier__max_depth=30, classifier__min_samples_leaf=2, classifier__min_samples_split=2, selector__k=10000; total time=   1.3s
[CV] END classifier__cri

In [273]:
top_words_dt = sorted_toxic_words_dt[:5]

In [274]:
top_words_dict['Decision Tree'] = top_words_dt

In [ ]:

print("\n" + "=" * 60)
print("СРАВНЕНИЕ ТОП-5 ТОКСИЧНЫХ СЛОВ")
print("-" * 60)

comparison_df = pd.DataFrame()
for model_name, words in top_words_dict.items():
    words_only = [word for word, _ in words]
    comparison_df[model_name] = words_only

print(comparison_df.to_string())



СРАВНЕНИЕ ТОП-5 ТОКСИЧНЫХ СЛОВ
------------------------------------------------------------
  Logistic Regression Naive Bayes Random Forest Decision Tree
0               хохлы      хохлов         хохол         хохол
1              хохлов        сука           год           год
2               нахуй       дебил       русский         блядь
3              пиздец       хохол         блядь          твой
4               блядь       русня         тупой       русский
